# PPO grid experiments on Colab

Run an independent **50,000-transition benchmark**, then **300,000 transitions each for seeds 0, 1, and 2** on level `1-1`. This notebook uses your current local source, including uncommitted changes. It contains no recorded training results.

**No new purchases:** The user has authorized the existing Colab Pro subscription and its available compute units for requested training. Never buy units, top up credits, upgrade, or start another subscription. Check the balance before connecting and the runtime’s consumption rate before training. Use the benchmark to estimate the suite’s remaining cost, leave a reserve, and monitor usage so you stop before exhaustion. The runner does not enforce a billing limit. If the existing allowance is insufficient, use verified free resources or the local runner.

1. On your computer, from the repository root, run `uv run python scripts/package_colab.py`.
2. Open this notebook in [Colab](https://colab.research.google.com/) with **File → Upload notebook**, without connecting.
3. After checking the existing balance, select a **T4 GPU with standard RAM**, connect, check its consumption rate, and run cells in order. This notebook does not allocate a GPU automatically; it defaults to `DEVICE = "cuda"` and checks CUDA availability. Run the benchmark before deciding whether all three experiments fit within the allowance.

Local fallback, from the repository root: `uv run python scripts/run_experiments.py --device mps --run-root runs/experiments/m4-baseline`. Use `--device cpu` on a machine without MPS.

GPU access, CPU count, RAM, and runtime duration vary; sessions may end early. With your separate Drive consent, checkpoints go directly to Google Drive every 50,000 transitions and at orderly interruption. Abrupt termination can lose progress since the latest completed write. Download results and use **Runtime → Disconnect and delete runtime** when finished. See the [Colab FAQ](https://research.google.com/colaboratory/faq.html).


## 1. Upload this checkout

Select only `runs/colab/mario-play.zip`, generated by the packaging command above. The archive includes source, configuration, and a SHA-256 manifest. It excludes `.git`, credentials, virtual environments, and training outputs. Only run code from a checkout you trust.


In [ ]:
import io
import os
import zipfile
from pathlib import Path

from google.colab import files

ROOT = Path("/content/mario-play")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one mario-play.zip source archive.")
archive_name, archive_bytes = next(iter(uploaded.items()))
if not archive_name.endswith(".zip"):
    raise ValueError("Expected the source ZIP made by scripts/package_colab.py.")
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for member in archive.infolist():
        if not (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve()):
            raise ValueError(f"Archive path leaves the source directory: {member.filename}")
    archive.extractall(ROOT)
if not (ROOT / "scripts/run_experiments.py").is_file():
    raise ValueError("Archive is missing scripts/run_experiments.py; rebuild it locally.")
os.chdir(ROOT)
print(f"Source extracted to {ROOT}")

## 2. Install into the Colab Python environment

Preserve the runtime's installed PyTorch build so its CUDA support stays intact. `pip` installs the project's other dependencies and the local package. No separate virtual environment is created.


In [ ]:
import subprocess
import sys
from importlib.metadata import version

constraints = Path("/content/mario-colab-constraints.txt")
constraints.write_text(f"torch=={version('torch')}\n")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--constraint", str(constraints), "-e", str(ROOT)],
    check=True,
)

## 3. Check the allocated hardware

Only reach this cell after checking the existing Pro allowance and the allocated runtime’s consumption rate. Hardware detection does not establish billing status. CUDA is the default for the T4 experiment; it must be available before training. Each device gets a separate output folder below. If no GPU was allocated, inspect the runtime and allowance before selecting a different runtime or intentionally changing `DEVICE` to `"cpu"`.


In [ ]:
import torch

DEVICE = "cuda"
print(
    {
        "python": sys.version,
        "torch": torch.__version__,
        "cpu_count": os.cpu_count(),
        "cuda_available": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
)
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable. Inspect the runtime and allowance before training.")

## 4. Mount Drive and select a durable experiment folder

Approve Google's Drive consent prompt yourself. Mounting Drive makes its files accessible to code in this notebook. Use a new folder for a new configuration; keep the same folder to resume an interrupted suite. Source identity is saved with the results. The runner rejects incompatible training settings when resuming.


In [ ]:
import json
import shutil

from google.colab import drive

drive.mount("/content/drive")
RUN_ROOT = Path(f"/content/drive/MyDrive/mario-experiments/ppo-grid-colab-{DEVICE}-v1")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
manifest = ROOT / "source_manifest.json"
saved_manifest = RUN_ROOT / "source_manifest.json"
if saved_manifest.exists() and saved_manifest.read_bytes() != manifest.read_bytes():
    raise ValueError("Source changed: choose a new RUN_ROOT to keep experiments comparable.")
shutil.copy2(manifest, saved_manifest)

base_command = [
    sys.executable,
    "scripts/run_experiments.py",
    "--device",
    DEVICE,
    "--run-root",
    str(RUN_ROOT),
    "--benchmark-steps",
    "50000",
    "--steps",
    "300000",
    "--seeds",
    "0",
    "1",
    "2",
    "--checkpoint-interval",
    "50000",
]
print(" ".join(base_command))

## 5. Run the independent 50k benchmark

This benchmark has its own run directory and does not warm-start the three seed experiments. Its summary records the hardware, elapsed time, throughput including evaluation/checkpoint overhead, and sampled/greedy evaluation metrics. Compare it with the same runner on your laptop before a longer run. Estimate the remaining suite’s compute units from measured time and the runtime’s unit rate; leave a reserve before proceeding.


In [ ]:
subprocess.run(base_command + ["--benchmark-only"], cwd=ROOT, check=True)
benchmark_summary = json.loads((RUN_ROOT / "benchmark/summary.json").read_text())
print(json.dumps(benchmark_summary, indent=2))

## 6. Run three independent 300k experiments

Run this cell after inspecting the benchmark. Repeating it resumes incomplete runs from their latest checkpoints and skips completed runs. Environments start new episodes after resume. To change hyperparameters, use a new `RUN_ROOT` and add repeated `--set` arguments, for example `"--set", "ppo.ent_coef=0.02"`.

Leave the notebook open and monitor the compute-unit balance while training. Interrupt and save results before the existing allowance is exhausted. If Colab disconnects, recheck the balance and rate before reconnecting, repeat setup and Drive mounting with the same source and folder, and rerun this cell. Use local compute if the allowance is insufficient. These are exploratory runs; 300k transitions do not guarantee a successful policy.


In [ ]:
subprocess.run(base_command + ["--skip-benchmark"], cwd=ROOT, check=True)

## 7. Inspect results

Each run has a `summary.json`, `metrics.csv`, TensorBoard events, and a `checkpoints/` directory. `held_out` compares sampled and greedy policies using fixed evaluation seeds. The root summary collects the benchmark and seed results. The per-run summaries below let you inspect each run independently.


In [ ]:
summaries = {
    path.parent.name: json.loads(path.read_text())
    for path in sorted(RUN_ROOT.glob("*/summary.json"))
}
for name, summary in summaries.items():
    print(
        json.dumps(
            {
                "run": name,
                "status": summary.get("status"),
                "steps": summary.get("global_step"),
                "training_steps_per_second": summary.get("training_steps_per_second"),
                "held_out": summary.get("held_out"),
            },
            indent=2,
        )
    )

## 8. Optional: download a local copy

The results already live in Drive. This cell additionally downloads a ZIP containing checkpoints, logs, summaries, and source provenance. Keep the Drive folder to resume later. When finished, use **Runtime → Disconnect and delete runtime** to release compute.


In [ ]:
result_zip = shutil.make_archive(
    "/content/mario-experiments-results",
    "zip",
    root_dir=RUN_ROOT.parent,
    base_dir=RUN_ROOT.name,
)
files.download(result_zip)